# P33 — AutoGen: aplicaciones de nueva generación mediante conversación multiagente

## 1. Título y paper

**Paper:** *AutoGen: Enabling Next-Gen LLM Applications via Multi-Agent Conversation*  
**Autoría:** Qingyun Wu, Gagan Bansal, Jieyu Zhang, Yiran Wu, y otros  
**Año y venue:** 2023 · arXiv:2308.08155  
**Nivel:** L4 · **Motor:** `autogen`  
**Ficha completa:** [`P33_autogen`](../../papers/foundational/P33_autogen/README.md)

**Hito:** El multiagente deja de ser una metáfora y pasa a ser un patrón de programación: agentes con rol que conversan hasta converger.

- [arXiv:2308.08155](https://arxiv.org/abs/2308.08155)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Un solo agente escribe y juzga su propio trabajo, así que arrastra sus propios puntos ciegos; y no había forma estándar de componer varios agentes con humanos en el bucle.
2. Ejecutar una implementación mínima de la propuesta: Agentes conversables y configurables —con o sin persona humana, con o sin ejecución de código— que se coordinan mediante mensajes, con patrones de conversación programables.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P13
- P30


## 4. Intuición

Quien escribe un texto es mal corrector de su propio texto: lee lo que quiso escribir. Poner a otro a revisarlo no es redundancia, es un punto de vista distinto sobre el mismo trabajo.


## 5. Concepto mínimo

```text
Un agente:     modelo → salida → (se juzga a sí mismo)

Multiagente:   planificador → programador → crítico → programador → …
               cada uno con su rol, su prompt y su objetivo
```

AutoGen lo formula como **conversación**: agentes configurables que se mandan mensajes, con o sin humano en el bucle, con o sin ejecución de código, y con patrones de conversación programables.


## 6. Código explicado

El motor compara una entrega de un solo agente con una conversación de tres roles.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('autogen', seed=7)['result']
print('UN SOLO AGENTE:')
show(r['un_solo_agente'])
print('\nMULTIAGENTE:')
for t in r['multiagente']['traza']:
    print(f"  [{t['rol']:<13}] {t['mensaje']}")

## 7. Predicción antes de ejecutar

1. ¿Qué fallo tiene el código del agente único?
2. ¿Por qué el crítico lo detecta y el propio autor no?
3. ¿Cuántas veces más caro sale en turnos?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
r = run_paper_lab('autogen', seed=7)['result']
print('un agente   → correcto:', r['un_solo_agente']['correcto'],
      '· turnos:', r['un_solo_agente']['turnos'])
print('multiagente → correcto:', r['multiagente']['correcto'],
      '· turnos:', r['multiagente']['turnos'])
print('coste relativo:', r['coste_relativo_en_turnos'], 'x')

## 9. Salida interpretable

La conversación encuentra el fallo y lo corrige, a cambio de **5× más turnos**. Esa es la cuenta que hay que hacer siempre: multiagente no es gratis y no es mejor por defecto. Hay que demostrar que el error que captura compensa el coste que añade.


## 10. Comentario pedagógico

El crítico funciona porque su **objetivo es distinto**: encontrar fallos, no producir código. Si los dos roles comparten prompt, modelo y objetivo, la crítica se vuelve ceremonial — es la versión multiagente de la sicofancia.


## 11. Error o anti-patrón deliberado

Anti-patrón: conversación sin criterio de parada. Dos agentes educados pueden felicitarse indefinidamente.


In [ ]:
turnos = 0
for _ in range(50):
    turnos += 1
    # cada uno espera que el otro cierre; nadie tiene autoridad para terminar
print(f'sin criterio de parada: {turnos} turnos y subiendo, sin converger')
print('cada turno es una llamada al modelo: el coste es lineal y no acotado')

## 12. Corrección

La corrección es dar autoridad de cierre y presupuesto:


In [ ]:
protocolo = {
    'criterio_de_exito': 'los tests pasan (verificable, no opinión)',
    'maximo_de_turnos': 8,
    'quien_cierra': 'el crítico, y solo con evidencia de ejecución',
    'deteccion_de_bucle': 'si dos turnos repiten el mismo mensaje, abortar',
    'escalamiento': 'al agotar el presupuesto, entregar a un humano con la traza',
}
show(protocolo)

## 13. Desafío guiado

Añade un cuarto rol (por ejemplo, un revisor de seguridad) y razona si aporta o solo encarece.


In [ ]:
roles = {'planificador': 'descompone', 'programador': 'implementa',
         'crítico': 'busca fallos', 'seguridad': 'busca riesgos'}
for n in range(2, 5):
    activos = list(roles)[:n]
    print(f'{n} roles {activos} → ~{n * 2} turnos mínimos')
print('\ncada rol nuevo debe justificar su coste con un tipo de error que SOLO él captura')

## 14. Desafío autónomo

Monta un sistema de dos y de cuatro agentes sobre la misma tarea de programación con tests. Ejecuta 30 veces cada configuración y compara tasa de éxito, turnos y coste. Comprueba si el multiagente gana a un agente único **bien construido**, que es la línea base honesta.


## 15. Evidencia de aprendizaje

Guarda ambas trazas, el coste relativo en turnos y tu protocolo con criterio de parada, detección de bucle y escalamiento.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P33_autogen/README.md) · evaluación formal: [`assessments/papers/P33_autogen.md`](../../assessments/papers/P33_autogen.md)


## 16. Cierre

Aquí se cierra el bloque de agentes. Todo lo que sigue —memoria compartida, protocolos entre proveedores, evaluación de trayectorias— vive en la frontera, con fecha.


## 17. Conexión con el siguiente hito

- P16

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
